# Chunking de Manifestações Longas

Comparação de duas configurações do RecursiveCharacterTextSplitter.

In [ ]:
!pip install -q langchain-text-splitters sentence-transformers scikit-learn pandas matplotlib

In [ ]:
MANIFESTACOES = [('M001', 'infraestrutura', 'calçada quebrada na rua central, dificultando a passagem de pedestres.'), ('M002', 'saúde', 'demora para conseguir consulta com clínico geral no posto do bairro.'), ('M003', 'infraestrutura', 'buraco enorme na Av. Brasil'), ('M004', 'segurança', 'falta de iluminação deixa a rua escura durante a noite.'), ('M005', 'educação', 'escola municipal está com salas sem ventiladores funcionando.'), ('M006', 'meio ambiente', 'lixo acumulado próximo ao córrego está atraindo insetos.'), ('M007', 'infraestrutura', 'semáforo do cruzamento principal apresenta defeito.'), ('M008', 'saúde', 'posto de saúde sem médico'), ('M009', 'segurança', 'moradores relatam aumento de furtos perto da praça.'), ('M010', 'educação', 'falta material escolar para as turmas do ensino fundamental.'), ('M011', 'meio ambiente', 'árvores precisam de poda na avenida do mercado.'), ('M012', 'infraestrutura', 'tampa de bueiro solta oferece risco para motociclistas.'), ('M013', 'saúde', 'medicamentos básicos estão em falta na unidade de saúde.'), ('M014', 'segurança', 'pedido de reforço de patrulhamento no bairro durante a madrugada.'), ('M015', 'educação', 'a biblioteca da escola precisa de computadores para os alunos.'), ('M016', 'meio ambiente', 'queimada em terreno baldio produz muita fumaça.'), ('M017', 'infraestrutura', 'asfalto todo esburacado da avenida principal'), ('M018', 'saúde', 'fila muito longa para atendimento odontológico no posto.'), ('M019', 'segurança', 'poste de iluminação pública apagado aumenta a sensação de insegurança.'), ('M020', 'educação', 'goteiras na sala de aula prejudicam as atividades em dias de chuva.'), ('M021', 'meio ambiente', 'descarte irregular de entulho em área verde do município.'), ('M022', 'saúde', 'falta atendimento no PSF'), ('M023', 'infraestrutura', 'chuva forte abriu uma cratera na avenida principal e os veículos precisam desviar.'), ('M024', 'segurança', 'iluminação pública insuficiente deixa a via perigosa à noite.'), ('M025', 'educação', 'a escola está sem professores substitutos para algumas disciplinas.'), ('M026', 'meio ambiente', 'acúmulo de resíduos em terreno próximo às casas precisa de limpeza.'), ('M027', 'infraestrutura', 'bueiro sem tampa na calçada pode causar acidentes.'), ('M028', 'saúde', 'unidade básica está sem remédios para pacientes que fazem tratamento contínuo.'), ('M029', 'segurança', 'moradores pedem presença policial mais frequente durante a noite.'), ('M030', 'educação', 'ventiladores quebrados deixam as salas de aula muito quentes.'), ('M031', 'saúde', 'lâmpada queimada na praça'), ('M032', 'meio ambiente', 'é necessária a poda de árvores que estão encostando nos fios.'), ('M033', 'infraestrutura', 'o semáforo do centro não está funcionando corretamente e causa congestionamento.'), ('M034', 'saúde', 'o atendimento médico do posto está demorando várias horas.'), ('M035', 'segurança', 'uma rua do bairro permanece completamente escura depois das 19 horas.'), ('M036', 'educação', 'faltam livros didáticos para parte dos estudantes da escola municipal. A situação tem causado transtornos frequentes à comunidade e os moradores solicitam providências, vistoria técnica, manutenção e acompanhamento do caso até que o problema seja resolvido de forma adequada. A situação tem causado transtornos frequentes à comunidade e os moradores solicitam providências, vistoria técnica, manutenção e acompanhamento do caso até que o problema seja resolvido de forma adequada. A situação tem causado transtornos frequentes à comunidade e os moradores solicitam providências, vistoria técnica, manutenção e acompanhamento do caso até que o problema seja resolvido de forma adequada.'), ('M037', 'meio ambiente', 'há descarte de lixo e entulho perto do rio, causando mau cheiro e atraindo animais. A situação tem causado transtornos frequentes à comunidade e os moradores solicitam providências, vistoria técnica, manutenção e acompanhamento do caso até que o problema seja resolvido de forma adequada. A situação tem causado transtornos frequentes à comunidade e os moradores solicitam providências, vistoria técnica, manutenção e acompanhamento do caso até que o problema seja resolvido de forma adequada. A situação tem causado transtornos frequentes à comunidade e os moradores solicitam providências, vistoria técnica, manutenção e acompanhamento do caso até que o problema seja resolvido de forma adequada.'), ('M038', 'infraestrutura', 'A avenida do bairro apresenta diversos problemas de pavimentação, com trechos desnivelados, buracos de tamanhos diferentes e pontos onde o asfalto já se desprendeu. Em dias de chuva, a água fica acumulada nessas áreas e dificulta a passagem de carros, motocicletas e ônibus. Moradores relatam que os veículos precisam reduzir muito a velocidade e fazer desvios, aumentando o risco de acidentes e causando transtornos para quem utiliza a via diariamente. Solicita-se avaliação técnica e reparo da pavimentação.'), ('M039', 'saúde', 'Os moradores desta região têm enfrentado dificuldades para conseguir atendimento na unidade básica de saúde. A equipe disponível não parece suficiente para a demanda atual, fazendo com que pacientes aguardem por muito tempo antes de serem atendidos. Pessoas idosas e cidadãos que precisam renovar receitas também relatam dificuldades para conseguir orientação. A comunidade solicita que a situação seja verificada e que sejam adotadas medidas para melhorar o fluxo de atendimento e a disponibilidade de profissionais.'), ('M040', 'meio ambiente', 'O terreno localizado próximo às residências vem recebendo descarte irregular de lixo, restos de construção e outros materiais. Com o passar dos dias, o volume de resíduos aumentou e passou a ocupar uma área considerável. Durante períodos de chuva, parte do material é levada para a rua e para a área próxima ao córrego. Moradores também relatam mau cheiro e presença de insetos e animais. Solicita-se fiscalização, limpeza do local e orientação aos responsáveis para evitar que novos descartes aconteçam.')]

In [ ]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.DataFrame(MANIFESTACOES, columns=["id","categoria","texto"])
longos = df[df.texto.str.len() > 500].copy()
longos[["id","texto"]].assign(caracteres=longos.texto.str.len())

In [ ]:
configuracoes = [
    (300, 30),
    (500, 100),
]

todos_chunks = {}
for chunk_size, chunk_overlap in configuracoes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    todos_chunks[(chunk_size, chunk_overlap)] = {
        row.id: splitter.split_text(row.texto)
        for _, row in longos.iterrows()
    }

for config, dados in todos_chunks.items():
    print("Configuração:", config)
    for doc_id, chunks in dados.items():
        print(doc_id, "->", len(chunks), "chunks")


## Influência do overlap

O overlap mantém parte do conteúdo do final de um chunk no início do seguinte. Isso reduz a quebra abrupta de contexto quando uma frase ou ideia se estende entre os limites. Um overlap pequeno tende a ser suficiente para preservar continuidade sem gerar muita redundância; um overlap maior aumenta a coesão, mas também cria mais conteúdo repetido e mais embeddings para armazenar.

In [ ]:
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
config = (500, 100)
linhas = []
for doc_id, chunks in todos_chunks[config].items():
    vetores = modelo.encode(chunks, normalize_embeddings=True)
    for i in range(len(chunks)-1):
        linhas.append({
            "Documento": doc_id,
            "Chunk A": i+1,
            "Chunk B": i+2,
            "Similaridade consecutiva": cosine_similarity(
                [vetores[i]], [vetores[i+1]]
            )[0,0]
        })
pd.DataFrame(linhas)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

config=(500,100)
registros=[]
for doc_id,chunks in todos_chunks[config].items():
    for i,chunk in enumerate(chunks):
        registros.append((doc_id,i,chunk))
V=modelo.encode([x[2] for x in registros], normalize_embeddings=True)
coords=PCA(n_components=2, random_state=42).fit_transform(V)

plt.figure(figsize=(8,6))
for doc_id in sorted(set(x[0] for x in registros)):
    mask=[x[0]==doc_id for x in registros]
    plt.scatter(coords[mask,0], coords[mask,1], label=doc_id)
plt.legend()
plt.title("Chunks no espaço 2D — PCA")
plt.show()

## Conclusão

A configuração (500, 100) tende a preservar melhor o contexto em manifestações longas porque combina chunks maiores com overlap suficiente para transportar informação entre limites. A configuração (300, 30) produz unidades menores, úteis para recuperação granular, mas pode fragmentar mais o sentido. A proximidade dos chunks de uma mesma manifestação no PCA é um indicativo visual, não uma garantia: PCA reduz a dimensionalidade e pode distorcer relações do espaço original.